In [ ]:
import kagglehub

path = kagglehub.dataset_download("muhammadshahidazeem/customer-churn-dataset")
print("Path to dataset files:", path)


In [ ]:
import os
print(os.listdir(path))

In [ ]:
# load and inspect data
import pandas as pd

df = pd.read_csv(os.path.join(path, "customer_churn_dataset-training-master.csv"))
print(df.shape)
df.head()
df.info()
df.isnull().sum()
df['Churn'].value_counts()

In [ ]:
# Drop identifier columns that carry no predictive signal
df = df.drop(columns=['CustomerID'], errors='ignore')

X = df.drop(columns=['Churn'])
y = df['Churn']

In [ ]:
print(X.dtypes)

In [ ]:
# Check which columns are categorical (text/object type)
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print(categorical_cols)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.summary()


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training vs Validation Accuracy')
plt.show()